# Stage 7 — Meta-Labeling and Bet Sizing

**AFML Chapters 3.6–3.8 (meta-labeling) and Chapter 10 (bet sizing)**

Phases: B (OOS primary predictions) → C (meta-labels) → D (meta-model) → E (bet sizing).

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                              f1_score, precision_score, recall_score,
                              accuracy_score, log_loss)

from src.meta_labeling import (
    generate_oos_predictions, make_meta_labels,
    build_meta_feature_matrix, generate_oos_meta_predictions,
    FEATURE_COLS_15,
)
from src.bet_sizing import get_signal, avg_active_signals, discrete_signal, build_daily_positions
from src.cross_validation import PurgedKFold

plt.rcParams.update({'figure.facecolor': 'white', 'axes.grid': True, 'font.size': 12})
import os; os.makedirs('../../reports/figures', exist_ok=True)
print("Imports OK")

## Phase A — Pre-flight checks

In [ ]:
md_df    = pd.read_parquet('../../data/processed/pooled_modelling.parquet')
labels   = pd.read_parquet('../../data/processed/meta_labels_pooled.parquet')
labels['bin'] = labels['label']  # make_meta_labels expects 'bin' column
close_panel = pd.read_parquet('../../data/processed/panel_ohlcv.parquet')['AdjClose'].unstack('ticker')
clean = close_panel[['NVDA']].rename(columns={'NVDA': 'Adj Close'})

import json
with open('../../models/best_params.json') as f:
    best_params = json.load(f)

RF_PARAMS = best_params['rf']['params']
NUM_TRIALS_TUNING = 50

print(f'Modelling dataset: {md_df.shape}')
print(f'Label dist: {md_df["label"].value_counts().to_dict()}')
print(f't1 NaN: {md_df["t1"].isna().sum()}')
print(f'Feature NaN: {md_df[FEATURE_COLS_15].isna().sum().sum()}')
print(f'RF params: {RF_PARAMS}')
majority_baseline = (md_df['label'] == 1).mean()
print(f'Majority baseline: {majority_baseline:.4f}')

In [ ]:
X  = md_df[FEATURE_COLS_15].copy()
y  = md_df['label'].copy()
w  = md_df['weight'].copy()
t1 = md_df['t1'].copy()
print(f'X: {X.shape}, y: {y.shape}')
print(f'Features: {FEATURE_COLS_15}')

## Phase B — OOS Primary Model Predictions

PurgedKFold loop with tuned RF. Every sample gets exactly one prediction from a model that never trained on it.

In [ ]:
oos_preds = generate_oos_predictions(
    X=X, y=y, sample_weight=w, t1=t1,
    clf_params=RF_PARAMS,
    n_splits=5, pct_embargo=0.01, random_state=42,
)
print(oos_preds.head())

In [ ]:
oos_acc = (oos_preds['pred_class'] == y).mean()
print(f'OOS primary accuracy : {oos_acc:.4f}')
print(f'Majority baseline    : {majority_baseline:.4f}')
print(f'Beats baseline       : {oos_acc > majority_baseline}')
print(f'Stage 4 reported CV  : 0.6280 (reference)')
print()
print('Side distribution:')
print(oos_preds['side'].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y, oos_preds['pred_class'], labels=[-1, 1])
ConfusionMatrixDisplay(cm, display_labels=['-1', '+1']).plot(ax=ax, colorbar=False)
ax.set_title('P_meta_1: OOS Primary Model\nPredicted vs Actual')
fig.tight_layout()
fig.savefig('../../reports/figures/pooled/P_meta_1_primary_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
oos_preds.to_parquet('../../data/processed/oos_predictions_pooled.parquet')
print('Saved oos_predictions_pooled.parquet')

## Phase C — Meta-Label Construction

`meta_label = 1` if `ret × side > 0` (trade in predicted direction was profitable).

In [ ]:
meta_labels_df = make_meta_labels(oos_preds=oos_preds, labels=labels)
print(meta_labels_df.head(10))

In [ ]:
dist = meta_labels_df['meta_label'].value_counts().sort_index()
meta_baseline = dist[1] / dist.sum()
print(f'Meta-label dist: {dist.to_dict()}')
print(f'Class-1 rate (meta-baseline): {meta_baseline:.4f}')

fig, ax = plt.subplots(figsize=(5, 4))
dist.plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'], edgecolor='black')
ax.set_xticks([0, 1])
ax.set_xticklabels(['0 (unprofitable)', '1 (profitable)'], rotation=0)
ax.set_ylabel('Count')
ax.set_title('P_meta_2: Meta-Label Distribution')
for i, v in enumerate(dist.values):
    ax.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
fig.tight_layout()
fig.savefig('../../reports/figures/pooled/P_meta_2_meta_label_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
X_meta, y_meta, w_meta, t1_meta = build_meta_feature_matrix(
    modelling_dataset=md_df,
    meta_labels=meta_labels_df,
    feature_cols=FEATURE_COLS_15,
)
print(f'X_meta: {X_meta.shape}')
print(f'y_meta dist: {y_meta.value_counts().to_dict()}')

In [ ]:
meta_labels_df = meta_labels_df.rename(columns={
    'pred_prob': 'oos_prob',
    'fold': 'oos_fold',
    'original_label': 'label',
})
meta_labels_df['ticker'] = md_df['ticker'].values
meta_labels_df['weight'] = md_df['weight'].values
meta_labels_df['t1']     = md_df['t1'].values
meta_labels_df.to_parquet('../../data/processed/meta_labels_pooled.parquet')
print('Saved meta_labels_pooled.parquet')

## Phase D — Meta-Model OOS Meta-Probabilities

Scoring: **F1** per AFML Snippet 9.1. Max depth=3 to prevent overfitting on 195 samples.

In [ ]:
META_CLF_PARAMS = {
    'n_estimators':     100,
    'max_depth':        3,
    'min_samples_leaf': 20,
    'max_features':     'sqrt',
    'class_weight':     'balanced',
}

oos_meta_preds = generate_oos_meta_predictions(
    X_meta=X_meta, y_meta=y_meta,
    w_meta=w_meta, t1_meta=t1_meta,
    meta_clf_params=META_CLF_PARAMS,
    n_splits=5, pct_embargo=0.01, random_state=42,
)
print(oos_meta_preds.head())

In [ ]:
# Per-fold metrics
cv_splitter = PurgedKFold(n_splits=5, t1=t1_meta, pct_embargo=0.01)
fold_metrics = []

for fold_idx, (train_idx, test_idx) in enumerate(cv_splitter.split(X_meta, y_meta)):
    X_tr, y_tr = X_meta.iloc[train_idx], y_meta.iloc[train_idx]
    X_te, y_te = X_meta.iloc[test_idx],  y_meta.iloc[test_idx]
    sw_tr = w_meta.iloc[train_idx].values
    sw_te = w_meta.iloc[test_idx].values

    clf = RandomForestClassifier(**{**META_CLF_PARAMS, 'random_state': 42})
    clf.fit(X_tr, y_tr, sample_weight=sw_tr)
    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te)[:, list(clf.classes_).index(1)]

    fold_metrics.append({
        'fold':      fold_idx,
        'n_test':    len(test_idx),
        'accuracy':  accuracy_score(y_te, y_pred, sample_weight=sw_te),
        'f1':        f1_score(y_te, y_pred, sample_weight=sw_te, zero_division=0),
        'precision': precision_score(y_te, y_pred, sample_weight=sw_te, zero_division=0),
        'recall':    recall_score(y_te, y_pred, sample_weight=sw_te, zero_division=0),
    })

fold_df = pd.DataFrame(fold_metrics)
print(fold_df.to_string(index=False))
print(f'\nMean F1  : {fold_df["f1"].mean():.4f} +/- {fold_df["f1"].std():.4f}')
print(f'Mean Acc : {fold_df["accuracy"].mean():.4f} +/- {fold_df["accuracy"].std():.4f}')
print(f'Meta-baseline: {meta_baseline:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(5)
bw = 0.25
ax.bar(x - bw, fold_df['f1'],        bw, label='F1',        color='#3498db')
ax.bar(x,      fold_df['precision'], bw, label='Precision',  color='#2ecc71')
ax.bar(x + bw, fold_df['recall'],    bw, label='Recall',     color='#e74c3c')
ax.axhline(meta_baseline, ls='--', color='black', lw=1.5, label=f'Baseline={meta_baseline:.3f}')
ax.set_xticks(x); ax.set_xticklabels([f'Fold {i}' for i in range(5)])
ax.set_ylabel('Score'); ax.set_title('P_meta_3: Meta-Model OOS Metrics per Fold')
ax.legend(); ax.set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig('../../reports/figures/pooled/P_meta_3_meta_fold_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(oos_meta_preds['meta_pred_prob'], bins=20, edgecolor='black', color='#3498db', alpha=0.8)
ax.axvline(0.5, ls='--', color='red', label='p=0.5 (no edge)')
ax.set_xlabel('P(meta_label=1)'); ax.set_ylabel('Count')
ax.set_title('P_meta_4: OOS Meta-Model Probability Distribution')
ax.legend()
fig.tight_layout()
fig.savefig('../../reports/figures/pooled/P_meta_4_meta_prob_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
oos_meta_preds.to_parquet('../../data/processed/meta_oos_predictions_pooled.parquet')
print('Saved meta_oos_predictions_pooled.parquet')

## Phase E — Bet Sizing and Daily Positions

Formula (AFML Snippet 10.1):
```
z = (p - 0.5) / sqrt(p*(1-p))
size = 2*Phi(z) - 1
signal = side * size
```
Step size = 0.2 (levels: 0, ±0.2, ±0.4, ±0.6, ±0.8, ±1.0).

In [ ]:
STEP_SIZE = 0.2

# 1. Raw signal from meta-probabilities
raw_signal = get_signal(
    side       = meta_labels_df['side'],
    meta_prob  = oos_meta_preds['meta_pred_prob'],
    num_classes= 2,
    step_size  = 0.0,
)

# 2. Average active signals across overlapping holding periods
t1_events = meta_labels_df['t1']  # t1 added from md_df in save cell
avg_pos = avg_active_signals(raw_signal, t1_events)

# 3. Discretise to step size
discrete_pos = discrete_signal(avg_pos, step_size=STEP_SIZE)

# 4. Expand to full daily index
daily_pos = build_daily_positions(discrete_pos, clean.index)

print(f'Raw signal: min={raw_signal.min():.3f}, max={raw_signal.max():.3f}, mean={raw_signal.mean():.3f}')
print(f'Discrete levels: {sorted(discrete_pos.unique())}')
print(f'Daily pos non-zero: {(daily_pos != 0).sum()} of {len(daily_pos)} days')

In [ ]:
# P_meta_5: signal distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(raw_signal.values, bins=20, edgecolor='black', color='#9b59b6', alpha=0.8)
axes[0].axvline(0, ls='--', color='red')
axes[0].set_title('Raw Signal (continuous)')
axes[0].set_xlabel('Signal [-1, 1]'); axes[0].set_ylabel('Count')

disc_vals = discrete_pos.value_counts().sort_index()
axes[1].bar(disc_vals.index.round(2).astype(str), disc_vals.values,
            color='#e67e22', edgecolor='black', alpha=0.8, width=0.6)
axes[1].set_title(f'Discretized Signal (step={STEP_SIZE})')
axes[1].set_xlabel('Signal Level'); axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

fig.suptitle('P_meta_5: Bet Size Distribution Before/After Discretization')
fig.tight_layout()
fig.savefig('../../reports/figures/pooled/P_meta_5_signal_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# P18: Position series on price chart
fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.plot(clean.index, clean['Adj Close'], color='steelblue', lw=1, label='NVDA Adj Close')
ax1.set_ylabel('Adj Close (USD, log)', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.set_yscale('log')
ax2.fill_between(daily_pos.index, 0, daily_pos.values,
                 where=daily_pos.values > 0, color='#2ecc71', alpha=0.4, label='Long')
ax2.fill_between(daily_pos.index, 0, daily_pos.values,
                 where=daily_pos.values < 0, color='#e74c3c', alpha=0.4, label='Short')
ax2.axhline(0, color='black', lw=0.8)
ax2.set_ylabel('Position [-1, 1]'); ax2.set_ylim(-1.3, 1.3)
ax1.set_xlabel('Date'); ax1.set_title('P18: Daily Position Series on NVDA Price')
lines1, lbl1 = ax1.get_legend_handles_labels()
lines2, lbl2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, lbl1 + lbl2, loc='upper left')
fig.tight_layout()
fig.savefig('../../reports/figures/pooled/P18_position_series.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# P_meta_6: event-level bet size distribution
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(raw_signal.abs().values, bins=15, edgecolor='black', color='#1abc9c', alpha=0.8)
ax.set_xlabel('|Signal| (Bet Size)'); ax.set_ylabel('Count')
ax.set_title('P_meta_6: Event-Level Bet Size Distribution')
fig.tight_layout()
fig.savefig('../../reports/figures/pooled/P_meta_6_bet_size_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# P_meta_7: active signals over time
n_active = pd.Series(0.0, index=clean.index)
for event_t, t1_val in zip(meta_labels_df.index, t1_events):
    if pd.isna(t1_val): continue
    mask = (clean.index >= event_t) & (clean.index <= t1_val)
    n_active[mask] += 1

fig, ax = plt.subplots(figsize=(14, 3))
ax.fill_between(n_active.index, n_active.values, color='#e67e22', alpha=0.6)
ax.set_ylabel('Active Signals'); ax.set_title('P_meta_7: Active Signals Over Time')
ax.set_xlabel('Date')
fig.tight_layout()
fig.savefig('../../reports/figures/pooled/P_meta_7_active_signals.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Max simultaneous active signals: {n_active.max():.0f}')
print(f'Mean active (when >0): {n_active[n_active>0].mean():.2f}')

In [ ]:
# Save positions parquet
positions_df = pd.DataFrame({'position': daily_pos}, index=daily_pos.index)
positions_df.to_parquet('../../data/processed/bet_sizes_pooled.parquet')
print('Saved bet_sizes_pooled.parquet')
print(f'Shape: {positions_df.shape}')
print(f'Non-zero days: {(positions_df["position"] != 0).sum()}')

## Stage 7 Summary

In [ ]:
print('=' * 60)
print('STAGE 7 SUMMARY')
print('=' * 60)
print(f'Primary OOS accuracy  : {oos_acc:.4f}')
print(f'Majority baseline     : {majority_baseline:.4f}')
print(f'Meta-baseline (pred 1): {meta_baseline:.4f}')
print(f'Meta-model mean F1    : {fold_df["f1"].mean():.4f} +/- {fold_df["f1"].std():.4f}')
print(f'Meta-model mean Acc   : {fold_df["accuracy"].mean():.4f} +/- {fold_df["accuracy"].std():.4f}')
print(f'Bet step size         : {STEP_SIZE}')
print(f'Active trading days   : {(daily_pos != 0).sum()} / {len(daily_pos)}')
print(f'Mean |position|       : {daily_pos.abs().mean():.4f}')